<a href="https://colab.research.google.com/github/VishalKumar-GitHub/AI-and-application/blob/main/ATS%20Score%20Analysis%20Using%20Job%20Description.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### End-to-End ATS Resume Scorer (Colorful Gradio Version)

This section provides the comprehensive workflow for analyzing your resume against a job description within an interactive, colorful Gradio web application. It simulates an Applicant Tracking System (ATS) to give you a compatibility score and actionable feedback, now with enhanced styling.

In [1]:
# Install necessary libraries
!pip install PyPDF2 python-multipart

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.7 MB/s eta 0:00:00


In [2]:
import gradio as gr
import os
import PyPDF2
import re

# --- Core Functions (re-defined for Gradio integration) ---

def extract_text_from_pdf_gradio(pdf_path):
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page_num in range(len(reader.pages)):
                page = reader.pages[page_num]
                text += page.extract_text() or ""
    except Exception as e:
        return f"Error extracting text from {pdf_path}: {e}"
    return text

def get_job_keywords_gradio(job_description_text):
    if not job_description_text.strip():
        return []

    if ',' in job_description_text:
        keywords = [k.strip() for k in job_description_text.split(',') if k.strip()]
    elif '\n' in job_description_text:
        keywords = [k.strip() for k in job_description_text.split('\n') if k.strip()]
    elif ';' in job_description_text:
        keywords = [k.strip() for k in job_description_text.split(';') if k.strip()]
    else:
        common_words = set(['a', 'an', 'the', 'is', 'are', 'and', 'or', 'to', 'in', 'for', 'with', 'on', 'of', 'we', 'you', 'your', 'from', 'as', 'by', 'at', 'this', 'that', 'will', 'be', 'have', 'has', 'our', 'and', 'or', 'not'])
        all_words = [word.strip().lower() for word in job_description_text.split() if word.strip()]
        keywords = list(set([word.capitalize() for word in all_words if word not in common_words and len(word) > 2]))

    keywords = sorted(list(set(keywords)))
    return keywords

def analyze_resume_for_ats_gradio(resume_text, job_description_keywords):
    report = {
        'overall_score': 0,
        'issues_count': 0,
        'content_category_score': 0,
        'sections_category_score': 0,
        'ats_essentials_category_score': 0,
        'tailoring_category_score': 0,
        'ats_parse_rate_score': 100,
        'ats_parse_rate_feedback': 'Great! We parsed 100% of your resume successfully using an industry-leading ATS.',
        'quantifiable_achievement_score': 0,
        'quantifiable_achievement_feedback': '',
        'repetition_score': 100,
        'repetition_feedback': 'No significant repetition issues detected (basic check).',
        'spelling_grammar_score': 100,
        'spelling_grammar_feedback': 'Spelling & Grammar: Automated check for this requires specialized NLP tools not included here. Please proofread carefully.',
        'action_verb_score': 0,
        'action_verb_feedback': '',
        'keyword_match_score': 0,
        'missing_keywords': [],
        'found_keywords': [],
        'sections_details': {'detected': []},
        'sections_feedback': [],
        'length_feedback': '',
        'length_score_contribution': 0
    }

    resume_text_lower = resume_text.lower()
    resume_words = resume_text_lower.split()
    resume_word_count = len(resume_words)

    # --- 1. Tailoring: Keyword Matching ---
    matched_keywords_count = 0
    current_found_keywords = []
    current_missing_keywords = []

    for keyword in job_description_keywords:
        # Simple check for whole word or phrase match
        if re.search(r'\b' + re.escape(keyword.lower()) + r'\b', resume_text_lower):
            matched_keywords_count += 1
            current_found_keywords.append(keyword)
        else:
            current_missing_keywords.append(keyword)

    report['found_keywords'] = current_found_keywords
    report['missing_keywords'] = current_missing_keywords

    if len(job_description_keywords) > 0:
        report['keyword_match_score'] = (matched_keywords_count / len(job_description_keywords)) * 100
    else:
        report['keyword_match_score'] = 0

    report['tailoring_category_score'] = report['keyword_match_score']
    if report['keyword_match_score'] < 70:
        report['issues_count'] += 1

    # --- 2. Sections: Section Presence ---
    common_sections = {
        'summary': ['summary', 'objective', 'profile'],
        'experience': ['experience', 'work experience', 'professional experience'],
        'education': ['education', 'academic history'],
        'skills': ['skills', 'technical skills', 'core competencies'],
        'projects': ['projects', 'portfolio'],
        'awards': ['awards', 'honors', 'achievements'],
        'certifications': ['certifications', 'licenses']
    }
    detected_sections_temp = {}
    for section_name, keywords in common_sections.items():
        for keyword in keywords:
            if re.search(r'\b' + re.escape(keyword) + r'\b', resume_text_lower):
                detected_sections_temp[section_name] = True
                break

    report['sections_details']['detected'] = list(detected_sections_temp.keys())
    if 'experience' not in detected_sections_temp:
        report['sections_feedback'].append("Consider adding an 'Experience' section with relevant job history.")
        report['issues_count'] += 1
    if 'education' not in detected_sections_temp:
        report['sections_feedback'].append("Ensure you have an 'Education' section.")
        report['issues_count'] += 1
    if 'skills' not in detected_sections_temp:
        report['sections_feedback'].append("An explicit 'Skills' section can help ATS identify your abilities.")
        report['issues_count'] += 1
    if len(detected_sections_temp) < 3:
         report['sections_feedback'].append("Consider organizing your resume with clear headings for better readability and ATS parsing.")
         report['issues_count'] += 1

    # Calculate sections_category_score
    if 'experience' in detected_sections_temp and 'education' in detected_sections_temp and 'skills' in detected_sections_temp:
        report['sections_category_score'] = 100
    elif len(detected_sections_temp) >= 3:
        report['sections_category_score'] = 80
    elif len(detected_sections_temp) >= 1:
        report['sections_category_score'] = 50
    else:
        report['sections_category_score'] = 20

    # --- 3. ATS Essentials: Resume Length ---
    if resume_word_count < 200:
        report['length_feedback'] = f"Your resume is quite short ({resume_word_count} words). Consider adding more details to showcase your experience and skills."
        report['length_score_contribution'] = 50
        report['issues_count'] += 1
    elif resume_word_count > 800:
        report['length_feedback'] = f"Your resume is quite long ({resume_word_count} words). Try to be concise and keep it to 1-2 pages for most roles. Focus on the most relevant achievements."
        report['length_score_contribution'] = 70
        report['issues_count'] += 1
    else:
        report['length_feedback'] = f"Your resume length ({resume_word_count} words) is generally good."
        report['length_score_contribution'] = 100

    report['ats_essentials_category_score'] = (report['length_score_contribution'] * 0.5 + report['ats_parse_rate_score'] * 0.5)

    # --- 4. Content: Action Verbs (Basic check) ---
    action_verbs = [
        "managed", "developed", "led", "created", "implemented", "analyzed", "optimized",
        "designed", "collaborated", "trained", "generated", "initiated", "launched",
        "mentored", "negotiated", "pioneered", "reduced", "resolved", "restructured",
        "supervised", "transformed", "unified", "validated", "achieved", "improved"
    ]
    action_verb_count = sum(1 for verb in action_verbs if re.search(r'\b' + re.escape(verb) + r'\b', resume_text_lower))

    # Score based on a reasonable density, for example, 5+ action verbs for a decent score
    if action_verb_count >= 15: # High count
        report['action_verb_score'] = 100
        report['action_verb_feedback'] = "Excellent! Strong use of action verbs throughout your resume."
    elif action_verb_count >= 8: # Moderate count
        report['action_verb_score'] = 75
        report['action_verb_feedback'] = "Good use of action verbs. Consider adding more to emphasize your contributions."
    elif action_verb_count >= 3: # Low count
        report['action_verb_score'] = 40
        report['action_verb_feedback'] = "Incorporate more strong action verbs at the start of your bullet points to highlight your contributions."
        report['issues_count'] += 1
    else: # Very low count
        report['action_verb_score'] = 10
        report['action_verb_feedback'] = "Significantly increase your use of action verbs to describe responsibilities and achievements."
        report['issues_count'] += 1


    # --- 5. Content: Quantifiable Achievements (Basic check) ---
    quantifiable_indicators = ["%", "dollars", "$", "million", "thousand", "reduced", "increased", "achieved", "improved", "launched", "developed", "managed", "oversaw"]
    quant_count = sum(1 for ind in quantifiable_indicators if re.search(r'\b' + re.escape(ind) + r'\b', resume_text_lower))
    # Look for numbers in proximity to quantifiable indicators, or just generally present
    numbers_found = bool(re.search(r'\b\d+[\.,]?\d*\s*(?:%|dollars|\$|million|thousand|k)?\b', resume_text_lower))

    if quant_count > 5 and numbers_found: # More robust check
        report['quantifiable_achievement_score'] = 100
        report['quantifiable_achievement_feedback'] = "Excellent! Your resume effectively highlights quantifiable achievements with strong impact."
    elif quant_count > 2 and numbers_found:
        report['quantifiable_achievement_score'] = 70
        report['quantifiable_achievement_feedback'] = "Good presence of quantifiable achievements. Try to add more specific metrics for greater impact."
    else:
        report['quantifiable_achievement_score'] = 20
        report['quantifiable_achievement_feedback'] = "Strengthen your resume by including more quantifiable achievements (e.g., 'increased sales by 15%', 'reduced costs by $10K', 'managed a team of 5')."
        report['issues_count'] += 1

    report['content_category_score'] = (report['action_verb_score'] * 0.5 + report['quantifiable_achievement_score'] * 0.5)

    # --- Overall Score (Weighted Average) ---
    report['overall_score'] = (
        report['content_category_score'] * 0.35 +
        report['sections_category_score'] * 0.25 +
        report['ats_essentials_category_score'] * 0.20 +
        report['tailoring_category_score'] * 0.20
    )

    return report

def generate_ats_report(pdf_file, job_description_input):
    if pdf_file is None:
        return "<p style='color:red;'>Please upload a resume PDF file.</p>"

    resume_path = pdf_file.name
    resume_content = extract_text_from_pdf_gradio(resume_path)

    if not resume_content:
        return "<p style='color:red;'>Could not extract text from the resume. Please ensure it's a searchable PDF.</p>"

    job_description_keywords = get_job_keywords_gradio(job_description_input)

    if not job_description_keywords:
        return "<p style='color:red;'>Please provide job description keywords.</p>"

    ats_report = analyze_resume_for_ats_gradio(resume_content, job_description_keywords)

    # Format the report for Gradio output with enhanced styling
    output_html = f"""
    <div style='background-color:#E8F5E9; padding:20px; border-radius:10px; border: 1px solid #C8E6C9;'>
        <h2 style='color:#2E7D32; text-align: center;'>&#10003; Comprehensive ATS Report</h2>
        <p style='font-size:1.1em; text-align: center;'><strong>Overall Estimated ATS Compatibility Score:</strong> <span style='font-size:1.4em; color:#1976D2; font-weight:bold;'>{ats_report['overall_score']:.2f}%</span></p>

        <h3 style='color:#388E3C;'>&#x1F50D; 1. Keyword Matching</h3>
        <p><strong>Match Score:</strong> <span style='color:#007BFF; font-weight:bold;'>{ats_report['keyword_match_score']:.2f}%</span></p>
        <p><strong>Keywords Matched:</strong> <span style='color:#4CAF50;'>{', '.join(ats_report['found_keywords']) if ats_report['found_keywords'] else 'None'}</span></p>
        <p><strong>Missing Keywords:</strong> <span style='color:#D32F2F;'>{', '.join(ats_report['missing_keywords']) if ats_report['missing_keywords'] else 'All specified keywords were found! Great job!'}</span></p>
        <p style='color:#555;'>To improve your keyword score, consider adding the missing keywords where relevant.</p>

        <h3 style='color:#388E3C;'>&#x1F4DA; 2. Resume Structure & Sections</h3>
        <p><strong>Detected Sections:</strong> <span style='color:#007BFF;'>{', '.join(ats_report['sections_details']['detected']) if ats_report['sections_details']['detected'] else 'No clear sections detected. Ensure clear headings are used.'}</span></p>
        <p><strong>Feedback:</strong></p>
        <ul style='list-style-type: disc; margin-left: 20px;'>
        {''.join([f"<li style='color:#555;'>{item}</li>" for item in ats_report['sections_feedback']]) if ats_report['sections_feedback'] else "<li style='color:#4CAF50;'>Looks good!</li>"}
        </ul>

        <h3 style='color:#388E3C;'>&#x1F4C3; 3. Resume Length</h3>
        <p><strong>Feedback:</strong> <span style='color:#555;'>{ats_report['length_feedback']}</span></p>

        <h3 style='color:#388E3C;'>&#x1F4AA; 4. Action Verbs</h3>
        <p><strong>Score (Basic):</strong> <span style='color:#007BFF; font-weight:bold;'>{ats_report['action_verb_score']:.2f}%</span></p>
        <p><strong>Feedback:</strong> <span style='color:#555;'>{ats_report['action_verb_feedback']}</span></p>

        <h3 style='color:#388E3C;'>&#x1F4B8; 5. Quantifiable Achievements</h3>
        <p><strong>Score (Basic):</strong> <span style='color:#007BFF; font-weight:bold;'>{ats_report['quantifiable_achievement_score']:.2f}%</span></p>
        <p><strong>Feedback:</strong> <span style='color:#555;'>{ats_report['quantifiable_achievement_feedback']}</span></p>

        <p style='margin-top:20px; font-style:italic; color:#777; text-align: center;'>Note: This is a simulated ATS report. For a full-fledged ATS, more advanced NLP and industry-specific metrics would be used.</p>
    </div>
    """
    return output_html

In [3]:
# --- Gradio Interface for ATS Scorer (with Colorful Theme) ---
# Apply the theme and CSS from the colorful dashboard example
custom_css = ".gradio-container { background-color: #F0F4C3; } .gr-button { background-color: #8BC34A !important; color: white !important; } .gr-text-input { border-color: #CDDC39 !important; } .gr-slider { --track-color: #FFEB3B !important;}"

iface_ats = gr.Interface(
    fn=generate_ats_report,
    inputs=[
        gr.File(label="Upload Your Resume (PDF only)", file_types=[".pdf"]),
        gr.Textbox(lines=5, label="Job Description / Keywords (comma-separated)", placeholder="e.g., Python, SQL, Machine Learning, Data Analysis, Cloud, Project Management")
    ],
    outputs=gr.HTML(label="ATS Compatibility Report"),
    title="<h1 style='text-align: center; color: #689F38;'>End-to-End ATS Resume Scorer</h1>",
    description="<p style='text-align: center; color: #7CB342;'>Upload your resume and paste a job description or keywords to get an instant Applicant Tracking System (ATS) compatibility score and detailed feedback, now with a vibrant look!</p>",
    allow_flagging="manual",
    flagging_dir="/tmp/gradio_flags",
    flagging_options=["Good Match", "Needs Improvement", "Irrelevant"],
    live=False,
    theme=gr.themes.Soft(), # Apply theme to the Interface constructor
    css=custom_css # Apply custom CSS to the Interface constructor
)

print("Launching Colorful Gradio ATS Scorer...")
# Pass theme and css to launch method to avoid deprecation warnings and apply custom styling
iface_ats.launch(share=True)
print("\nYour Colorful Gradio ATS Scorer is launching. Look for a public URL in the output above.")

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Launching Colorful Gradio ATS Scorer...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4c1b7c5df0e712203f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



Your Colorful Gradio ATS Scorer is launching. Look for a public URL in the output above.
